<a href="https://colab.research.google.com/github/manojgundumogula55-commits/Digital-Library-Usage-And-Resource-Analytics/blob/main/Digital_Library_Usage_Analytics/03_Data_Aggregation_Representation_and_Analysis/Data_Aggregation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Digital Library Usage Analysis
## Data Aggregation, Representation and Analysis
### This notebook covers the Data Aggregation, Data Representation and Data Analysis stages of the project. It uses the cleaned dataset produced in Stage 2.

# 1. Import Libraries

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [ ]:
# 2. Load the Cleaned Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name, parse_dates=["access_datetime"])

print("Cleaned dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Saving library_usage_raw.csv to library_usage_raw.csv
Cleaned dataset loaded successfully.
Rows: 1500
Columns: 15


In [ ]:
## 2.1 Quick Check of the Cleaned Data
### The cleaned dataset should have no duplicates and no missing session times.

In [ ]:
print("Duplicate records:", df.duplicated().sum())
print("Missing session times:", df["session_minutes"].isna().sum())
print("Missing ratings (retained as not reported):", df["rating"].isna().sum())

display(df.head())

Duplicate records: 29
Missing session times: 67
Missing ratings (retained as not reported): 89


,access_id,user_id,user_type,department,resource_id,resource_title,resource_type,publisher,publication_year,access_datetime,session_minutes,device,downloaded,rating,access_mode
0,A00001,U1675,Undergraduate,Mechanical,R070,SpringerLink Vol.1,Database,IEEE,2015,2025-03-19 23:30:00,33.6,Laptop,0,4.0,Remote
1,A00002,U1448,Undergraduate,ECE,R068,PhD Thesis Vol.3,Thesis,Wiley,2008,2025-03-26 14:18:00,26.9,Laptop,1,4.0,Remote
2,A00003,U1015,Faculty,Civil,R023,Finance Vol.2,Journal,Pearson,2005,2025-01-27 07:54:00,60.3,Campus PC,0,5.0,Remote
3,A00004,U1124,Undergraduate,Sciences,R049,Finance Vol.2,Journal,NPTEL,2011,2025-12-16 12:39:00,42.0,Laptop,1,2.0,On-Campus
4,A00005,U1298,Postgraduate,Sciences,R056,Signal Processing Vol.5,Journal,Wiley,2021,2025-09-20 11:53:00,8.0,Laptop,0,4.0,Remote


In [ ]:
## 2.2 Reference Orders
### These orders keep months and weekdays in calendar order in every table.

In [ ]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

In [ ]:
# 3. Data Aggregation
### The aggregation tables summarise individual access records into counts, averages and rates. They are built from the cleaned dataset `df`.

## 3.1 Overall Usage Summary

In [ ]:
overall_summary = pd.Series({
    "Total accesses": len(df),
    "Unique users": df["user_id"].nunique(),
    "Unique resources accessed": df["resource_id"].nunique(),
    "Average session (minutes)": round(df["session_minutes"].mean(), 1),
    "Total hours spent": round(df["session_minutes"].sum() / 60, 1),
    "Download rate (%)": round(df["downloaded"].mean() * 100, 1),
    "Average rating": round(df["rating"].mean(), 2),
    "Remote access (%)": round((df["access_mode"] == "Remote").mean() * 100, 1)
}, name="value")

display(overall_summary.to_frame())

,value
Total accesses,1500.00
Unique users,622.00
Unique resources accessed,71.00
Average session (minutes),39.50
Total hours spent,943.70
Download rate (%),32.90
Average rating,3.77
Remote access (%),40.60


In [ ]:
## 3.2 Resource-wise Aggregation

In [ ]:
resource_agg = (
    df.groupby(["resource_id", "resource_title", "resource_type"])
      .agg(
          access_count=("access_id", "count"),
          unique_users=("user_id", "nunique"),
          avg_session=("session_minutes", "mean"),
          downloads=("downloaded", "sum"),
          avg_rating=("rating", "mean"),
          rating_count=("rating", "count")
      )
      .round(2)
      .reset_index()
      .sort_values("access_count", ascending=False)
)

display(resource_agg.head(15))

,resource_id,resource_title,resource_type,access_count,unique_users,avg_session,downloads,avg_rating,rating_count
62,R070,SpringerLink Vol.1,Database,129,114,29.46,45,3.82,121
59,R067,Data Science Vol.2,E-Book,93,85,39.77,28,3.71,87
32,R038,Marketing Vol.2,E-Book,84,76,39.05,35,3.78,79
10,R013,Marketing Vol.3,E-Book,75,69,44.08,30,3.72,69
65,R073,Thermodynamics Vol.1,E-Book,63,59,44.14,20,3.98,59
55,R063,Chemistry Vol.1,Journal,57,51,45.90,25,3.53,57
30,R036,JSTOR Vol.3,Database,54,50,30.72,16,3.96,51
15,R019,ScienceDirect Vol.4,Database,52,52,66.50,16,3.67,48
35,R042,Finance Vol.1,Journal,50,48,25.65,11,3.71,48
19,R023,Finance Vol.2,Journal,49,45,53.79,17,3.89,46


In [ ]:
### Resource Type Aggregation

In [ ]:
type_agg = (
    df.groupby("resource_type")
      .agg(
          access_count=("access_id", "count"),
          unique_users=("user_id", "nunique"),
          avg_session=("session_minutes", "mean"),
          download_rate=("downloaded", "mean"),
          avg_rating=("rating", "mean")
      )
      .round(2)
      .sort_values("access_count", ascending=False)
)

type_agg["share_percent"] = (type_agg["access_count"] / type_agg["access_count"].sum() * 100).round(1)
type_agg["download_rate"] = (type_agg["download_rate"] * 100).round(1)

display(type_agg)

,access_count,unique_users,avg_session,download_rate,avg_rating,share_percent
resource_type,,,,,,
E-Book,549,385,37.86,34.0,3.75,36.6
Database,400,299,39.10,32.0,3.83,26.7
Journal,392,296,43.55,32.0,3.73,26.1
Thesis,127,118,29.78,31.0,3.80,8.5
Video,32,31,61.43,44.0,3.69,2.1


In [ ]:
### Publisher Aggregation

In [ ]:
publisher_agg = (
    df.groupby("publisher")
      .agg(
          access_count=("access_id", "count"),
          resources=("resource_id", "nunique"),
          avg_rating=("rating", "mean")
      )
      .round(2)
      .sort_values("access_count", ascending=False)
)

display(publisher_agg)

,access_count,resources,avg_rating
publisher,,,
IEEE,387,14,3.70
Pearson,296,11,3.82
McGraw Hill,190,11,3.80
NPTEL,186,8,3.77
Wiley,164,10,3.72
Springer,130,8,3.74
Taylor & Francis,125,6,3.85
Elsevier,6,3,4.17


In [ ]:
## 3.3 User and Department-wise Aggregation

In [ ]:
department_agg = (
    df.groupby("department")
      .agg(
          access_count=("access_id", "count"),
          unique_users=("user_id", "nunique"),
          avg_session=("session_minutes", "mean"),
          download_rate=("downloaded", "mean")
      )
      .round(2)
      .sort_values("access_count", ascending=False)
)

department_agg["accesses_per_user"] = (department_agg["access_count"] / department_agg["unique_users"]).round(2)
department_agg["download_rate"] = (department_agg["download_rate"] * 100).round(1)

display(department_agg)

,access_count,unique_users,avg_session,download_rate,accesses_per_user
department,,,,,
CSE,265,219,40.67,31.0,1.21
Mechanical,250,217,44.21,36.0,1.15
ECE,249,198,44.04,36.0,1.26
Management,242,202,42.91,37.0,1.20
Civil,224,188,32.56,28.0,1.19
Sciences,220,189,33.08,31.0,1.16
cse,29,28,27.40,17.0,1.04
ece,21,20,35.68,29.0,1.05


In [ ]:
### User Type Aggregation

In [ ]:
user_type_agg = (
    df.groupby("user_type")
      .agg(
          access_count=("access_id", "count"),
          unique_users=("user_id", "nunique"),
          avg_session=("session_minutes", "mean"),
          download_rate=("downloaded", "mean"),
          avg_rating=("rating", "mean")
      )
      .round(2)
      .sort_values("access_count", ascending=False)
)

user_type_agg["download_rate"] = (user_type_agg["download_rate"] * 100).round(1)

display(user_type_agg)

,access_count,unique_users,avg_session,download_rate,avg_rating
user_type,,,,,
Undergraduate,793,473,37.91,33.0,3.77
Postgraduate,370,287,40.32,32.0,3.69
Faculty,174,157,47.16,29.0,3.90
Researcher,137,123,38.61,41.0,3.89
UNDERGRADUATE,15,14,31.44,27.0,3.40
POSTGRADUATE,6,5,28.34,50.0,2.80
FACULTY,4,4,31.68,50.0,2.50
RESEARCHER,1,1,28.50,100.0,3.00


In [ ]:
## 3.4 Time-based Aggregation

In [ ]:
df["access_datetime"] = pd.to_datetime(df["access_datetime"], errors="coerce")

df["month"] = df["access_datetime"].dt.month_name()
df["weekday"] = df["access_datetime"].dt.day_name()
df["hour"] = df["access_datetime"].dt.hour
monthly_agg = (
    df.groupby("month")
      .agg(
          access_count=("access_id", "count"),
          unique_users=("user_id", "nunique"),
          avg_session=("session_minutes", "mean")
      )
      .round(2)
      .reindex(month_order)
)

monthly_agg["change_from_previous_month_%"] = monthly_agg["access_count"].pct_change().mul(100).round(1)

display(monthly_agg)

,access_count,unique_users,avg_session,change_from_previous_month_%
month,,,,
January,99,91,26.85,NaN
February,117,108,69.60,18.2
March,171,151,35.73,46.2
April,188,159,30.55,9.9
May,77,73,50.40,-59.0
June,52,49,25.80,-32.5
July,94,81,35.26,80.8
August,148,134,36.59,57.4
September,163,145,47.47,10.1


In [ ]:
### Weekday Aggregation

In [ ]:
weekday_agg = (
    df.groupby("weekday")
      .agg(
          access_count=("access_id", "count"),
          avg_session=("session_minutes", "mean")
      )
      .round(2)
      .reindex(day_order)
)

display(weekday_agg)

,access_count,avg_session
weekday,,
Monday,214,46.85
Tuesday,215,30.35
Wednesday,219,38.19
Thursday,211,43.66
Friday,216,47.34
Saturday,204,33.81
Sunday,206,36.90


In [ ]:
### Hourly Aggregation

In [ ]:
hourly_agg = (
    df.groupby("hour")
      .agg(
          access_count=("access_id", "count"),
          avg_session=("session_minutes", "mean")
      )
      .round(2)
)

display(hourly_agg)

,access_count,avg_session
hour,,
0.0,17,30.82
1.0,7,23.79
2.0,10,31.09
3.0,11,29.20
4.0,11,17.89
5.0,13,122.63
6.0,21,22.53
7.0,33,30.84
8.0,45,53.52


In [ ]:
## 3.5 Device and Access Mode Aggregation

In [ ]:
device_agg = (
    df.groupby("device")
      .agg(
          access_count=("access_id", "count"),
          avg_session=("session_minutes", "mean"),
          download_rate=("downloaded", "mean")
      )
      .round(2)
      .sort_values("access_count", ascending=False)
)
device_agg["download_rate"] = (device_agg["download_rate"] * 100).round(1)

display(device_agg)

,access_count,avg_session,download_rate
device,,,
Laptop,672,40.79,33.0
Mobile,543,40.11,30.0
Campus PC,264,35.98,38.0


In [ ]:
access_mode_agg = (
    df.groupby("access_mode")
      .agg(
          access_count=("access_id", "count"),
          avg_session=("session_minutes", "mean"),
          download_rate=("downloaded", "mean")
      )
      .round(2)
)
access_mode_agg["download_rate"] = (access_mode_agg["download_rate"] * 100).round(1)

display(access_mode_agg)

,access_count,avg_session,download_rate
access_mode,,,
On-Campus,891,40.26,33.0
Remote,609,38.40,33.0


In [ ]:
# 4. Data Representation
### The representation tables arrange the aggregated data as cross-tabulations and pivot tables, which are the tables used directly for charts in Stage 4.

## 4.1 Department vs Resource Type

In [ ]:
dept_type_table = pd.crosstab(df["department"], df["resource_type"], margins=True, margins_name="Total")

display(dept_type_table)

resource_type,Database,E-Book,Journal,Thesis,Video,Total
department,,,,,,
cse,9,8,7,3,2,29
CSE,77,96,70,14,8,265
Civil,63,82,58,18,3,224
ECE,61,89,74,21,4,249
Management,66,91,51,29,5,242
Mechanical,67,93,68,18,4,250
Sciences,52,82,60,20,6,220
ece,5,8,4,4,0,21
Total,400,549,392,127,32,1500


In [ ]:
### Department vs Resource Type (Row Percentage)

In [ ]:
dept_type_percent = (
    pd.crosstab(df["department"], df["resource_type"], normalize="index") * 100
).round(1)

display(dept_type_percent)

resource_type,Database,E-Book,Journal,Thesis,Video
department,,,,,
cse,31.0,27.6,24.1,10.3,6.9
CSE,29.1,36.2,26.4,5.3,3.0
Civil,28.1,36.6,25.9,8.0,1.3
ECE,24.5,35.7,29.7,8.4,1.6
Management,27.3,37.6,21.1,12.0,2.1
Mechanical,26.8,37.2,27.2,7.2,1.6
Sciences,23.6,37.3,27.3,9.1,2.7
ece,23.8,38.1,19.0,19.0,0.0


In [ ]:
## 4.2 User Type vs Resource Type

In [ ]:
user_type_table = pd.crosstab(df["user_type"], df["resource_type"], margins=True, margins_name="Total")

display(user_type_table)

resource_type,Database,E-Book,Journal,Thesis,Video,Total
user_type,,,,,,
FACULTY,2,0,2,0,0,4
Faculty,57,67,31,16,3,174
POSTGRADUATE,1,4,1,0,0,6
Postgraduate,99,140,92,32,7,370
RESEARCHER,0,0,1,0,0,1
Researcher,47,45,37,6,2,137
UNDERGRADUATE,8,3,3,1,0,15
Undergraduate,186,290,225,72,20,793
Total,400,549,392,127,32,1500


In [ ]:
## 4.3 Month vs Resource Type

In [ ]:
month_type_table = pd.crosstab(df["month"], df["resource_type"]).reindex(month_order)

display(month_type_table)

resource_type,Database,E-Book,Journal,Thesis,Video
month,,,,,
January,25,44,22,6,2
February,39,32,36,8,2
March,51,50,47,22,1
April,55,59,50,19,5
May,15,25,24,10,3
June,13,19,16,3,1
July,26,34,22,8,4
August,28,65,35,16,4
September,40,62,48,12,1


In [ ]:
## 4.4 Weekday vs Hour Usage Matrix

In [ ]:
weekday_hour_matrix = df.pivot_table(
    index="weekday",
    columns="hour",
    values="access_id",
    aggfunc="count",
    fill_value=0
).reindex(day_order)

display(weekday_hour_matrix)

hour,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0
weekday,,,,,,,,,,,,,,,,,,,,,,,,
Monday,1,1,0,2,0,2,3,8,5,11,15,22,10,8,14,15,17,12,16,17,14,11,4,6
Tuesday,3,0,0,2,1,3,4,3,6,12,15,16,15,9,15,11,13,14,14,12,18,18,4,7
Wednesday,3,1,3,2,3,2,2,4,6,16,18,11,15,5,13,19,18,15,11,10,15,15,9,3
Thursday,3,2,1,1,0,1,3,3,6,6,12,18,17,12,14,19,24,12,12,15,11,8,7,4
Friday,2,2,2,2,4,1,0,5,5,16,17,12,4,18,14,10,18,11,8,18,20,16,7,4
Saturday,4,1,0,2,3,1,5,7,8,9,15,8,17,7,12,14,11,18,11,11,13,21,4,2
Sunday,1,0,4,0,0,3,4,3,9,16,11,14,7,9,18,15,11,16,7,14,16,13,8,7


In [ ]:
## 4.5 Device vs Access Mode

In [ ]:
device_mode_table = pd.crosstab(df["device"], df["access_mode"], margins=True, margins_name="Total")

display(device_mode_table)

access_mode,On-Campus,Remote,Total
device,,,
Campus PC,156,108,264
Laptop,399,273,672
Mobile,322,221,543
Total,877,602,1479


In [ ]:
## 4.6 Average Session Time by Department and Resource Type

In [ ]:
session_pivot = df.pivot_table(
    index="department",
    columns="resource_type",
    values="session_minutes",
    aggfunc="mean"
).round(1)

display(session_pivot)

resource_type,Database,E-Book,Journal,Thesis,Video
department,,,,,
cse,20.6,32.3,24.4,45.9,20.4
CSE,48.1,40.6,35.7,33.5,27.2
Civil,42.9,25.7,31.5,27.3,51.0
ECE,27.1,25.8,69.7,32.1,274.4
Management,46.2,53.3,30.6,27.9,22.1
Mechanical,42.0,50.5,43.4,28.1,28.0
Sciences,25.8,28.7,47.6,25.6,32.3
ece,39.6,39.3,16.1,43.9,NaN


In [ ]:
# 5. Data Analysis
### Each analysis question below uses the aggregation and representation tables created above.

## 5.1 Which resources are used most?

In [ ]:
top_10_resources = resource_agg.head(10)[["resource_id", "resource_title", "resource_type", "access_count", "unique_users"]]

display(top_10_resources.reset_index(drop=True))

top_10_share = resource_agg.head(10)["access_count"].sum() / len(df) * 100
print(f"The top 10 resources account for {top_10_share:.1f}% of all accesses.")

,resource_id,resource_title,resource_type,access_count,unique_users
0,R070,SpringerLink Vol.1,Database,129,114
1,R067,Data Science Vol.2,E-Book,93,85
2,R038,Marketing Vol.2,E-Book,84,76
3,R013,Marketing Vol.3,E-Book,75,69
4,R073,Thermodynamics Vol.1,E-Book,63,59
5,R063,Chemistry Vol.1,Journal,57,51
6,R036,JSTOR Vol.3,Database,54,50
7,R019,ScienceDirect Vol.4,Database,52,52
8,R042,Finance Vol.1,Journal,50,48
9,R023,Finance Vol.2,Journal,49,45


The top 10 resources account for 47.1% of all accesses.


In [ ]:
## 5.2 Which resource types are most popular?

In [ ]:
display(type_agg[["access_count", "share_percent", "avg_session", "download_rate"]])

print("Most accessed resource type:", type_agg["access_count"].idxmax())
print("Highest download rate:", type_agg["download_rate"].idxmax())
print("Longest average session:", type_agg["avg_session"].idxmax())

,access_count,share_percent,avg_session,download_rate
resource_type,,,,
E-Book,549,36.6,37.86,34.0
Database,400,26.7,39.10,32.0
Journal,392,26.1,43.55,32.0
Thesis,127,8.5,29.78,31.0
Video,32,2.1,61.43,44.0


Most accessed resource type: E-Book
Highest download rate: Video
Longest average session: Video


In [ ]:
## 5.3 Which resources are underused?
### Resources in the lowest 10% of access counts are treated as underused and can be reviewed for licence renewal.

In [ ]:
threshold = resource_agg["access_count"].quantile(0.10)

underused = resource_agg[resource_agg["access_count"] <= threshold][
    ["resource_id", "resource_title", "resource_type", "access_count", "unique_users"]
].sort_values("access_count")

print("Underused threshold (accesses):", threshold)
print("Number of underused resources:", len(underused))

display(underused.reset_index(drop=True))

Underused threshold (accesses): 2.0
Number of underused resources: 10


,resource_id,resource_title,resource_type,access_count,unique_users
0,R052,Signal Processing Vol.1,Journal,1,1
1,R029,Programming Vol.1,E-Book,1,1
2,R010,Structural Design Vol.1,E-Book,1,1
3,R015,IEEE Xplore Vol.2,Database,1,1
4,R072,NPTEL Lecture Vol.4,Video,1,1
5,R060,MTech Thesis Vol.5,Thesis,1,1
6,R037,Thermodynamics Vol.5,E-Book,2,2
7,R033,Chemistry Vol.1,Journal,2,2
8,R003,JSTOR Vol.4,Database,2,2
9,R031,Thermodynamics Vol.4,E-Book,2,2


In [ ]:
### Resources Accessed by Only One User

In [ ]:
single_user = resource_agg[resource_agg["unique_users"] == 1][
    ["resource_id", "resource_title", "resource_type", "access_count"]
]

print("Resources used by only one user:", len(single_user))

display(single_user.reset_index(drop=True))

Resources used by only one user: 6


,resource_id,resource_title,resource_type,access_count
0,R015,IEEE Xplore Vol.2,Database,1
1,R010,Structural Design Vol.1,E-Book,1
2,R029,Programming Vol.1,E-Book,1
3,R052,Signal Processing Vol.1,Journal,1
4,R060,MTech Thesis Vol.5,Thesis,1
5,R072,NPTEL Lecture Vol.4,Video,1


In [ ]:
## 5.4 Which departments and user types use the library most?

In [ ]:
display(department_agg[["access_count", "unique_users", "accesses_per_user", "avg_session"]])

print("Most active department:", department_agg["access_count"].idxmax())
print("Least active department:", department_agg["access_count"].idxmin())
print("Highest accesses per user:", department_agg["accesses_per_user"].idxmax())

,access_count,unique_users,accesses_per_user,avg_session
department,,,,
CSE,265,219,1.21,40.67
Mechanical,250,217,1.15,44.21
ECE,249,198,1.26,44.04
Management,242,202,1.20,42.91
Civil,224,188,1.19,32.56
Sciences,220,189,1.16,33.08
cse,29,28,1.04,27.40
ece,21,20,1.05,35.68


Most active department: CSE
Least active department: ece
Highest accesses per user: ECE


In [ ]:
user_share = (user_type_agg["access_count"] / user_type_agg["access_count"].sum() * 100).round(1)

display(user_share.rename("share_percent").to_frame())

,share_percent
user_type,
Undergraduate,52.9
Postgraduate,24.7
Faculty,11.6
Researcher,9.1
UNDERGRADUATE,1.0
POSTGRADUATE,0.4
FACULTY,0.3
RESEARCHER,0.1


In [ ]:
## 5.5 When is usage highest?

In [ ]:
peak_month = monthly_agg["access_count"].idxmax()
low_month = monthly_agg["access_count"].idxmin()
peak_day = weekday_agg["access_count"].idxmax()
peak_hour = hourly_agg["access_count"].idxmax()

print("Peak month:", peak_month, "-", monthly_agg.loc[peak_month, "access_count"], "accesses")
print("Lowest month:", low_month, "-", monthly_agg.loc[low_month, "access_count"], "accesses")
print("Busiest weekday:", peak_day)
print("Busiest hour:", f"{peak_hour}:00")

top_3_months = monthly_agg["access_count"].nlargest(3)
print("\nTop 3 months:")
display(top_3_months.to_frame())

Peak month: April - 188 accesses
Lowest month: June - 52 accesses
Busiest weekday: Wednesday
Busiest hour: 16.0:00

Top 3 months:


,access_count
month,
April,188
October,188
March,171


In [ ]:
### Peak Weekday and Hour Combinations

In [ ]:
peak_slots = (
    weekday_hour_matrix.stack()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
peak_slots.columns = ["weekday", "hour", "access_count"]

display(peak_slots)

,weekday,hour,access_count
0,Thursday,16.0,24
1,Monday,11.0,22
2,Saturday,21.0,21
3,Friday,20.0,20
4,Wednesday,15.0,19
5,Thursday,15.0,19
6,Thursday,11.0,18
7,Wednesday,16.0,18
8,Tuesday,21.0,18
9,Tuesday,20.0,18


In [ ]:
## 5.6 How long do users stay, and what do they download?

In [ ]:
print("Average session by user type:")
display(user_type_agg[["avg_session"]])

print("Average session by device:")
display(device_agg[["avg_session"]])

print("Download rate by resource type (%):")
display(type_agg[["download_rate"]])

Average session by user type:


,avg_session
user_type,
Undergraduate,37.91
Postgraduate,40.32
Faculty,47.16
Researcher,38.61
UNDERGRADUATE,31.44
POSTGRADUATE,28.34
FACULTY,31.68
RESEARCHER,28.50


Average session by device:


,avg_session
device,
Laptop,40.79
Mobile,40.11
Campus PC,35.98


Download rate by resource type (%):


,download_rate
resource_type,
E-Book,34.0
Database,32.0
Journal,32.0
Thesis,31.0
Video,44.0


In [ ]:
## 5.7 Remote vs On-Campus Usage

In [ ]:
display(access_mode_agg)

remote_share = access_mode_agg.loc["Remote", "access_count"] / access_mode_agg["access_count"].sum() * 100
print(f"Remote access makes up {remote_share:.1f}% of all accesses.")

,access_count,avg_session,download_rate
access_mode,,,
On-Campus,891,40.26,33.0
Remote,609,38.40,33.0


Remote access makes up 40.6% of all accesses.


In [ ]:
## 5.8 Which resources are rated highest?
### Only resources with at least 10 ratings are compared, so a single rating cannot decide the ranking.

In [ ]:
rated_resources = resource_agg[resource_agg["rating_count"] >= 10]

top_rated = rated_resources.sort_values("avg_rating", ascending=False).head(10)[
    ["resource_id", "resource_title", "resource_type", "avg_rating", "rating_count"]
]

display(top_rated.reset_index(drop=True))

print("Average rating by resource type:")
display(type_agg[["avg_rating"]])

,resource_id,resource_title,resource_type,avg_rating,rating_count
0,R077,Chemistry Vol.1,Journal,4.36,11
1,R054,ScienceDirect Vol.1,Database,4.12,16
2,R021,JSTOR Vol.3,Database,4.07,27
3,R001,JSTOR Vol.3,Database,4.07,14
4,R079,Project Report Vol.5,Thesis,4.00,45
5,R073,Thermodynamics Vol.1,E-Book,3.98,59
6,R036,JSTOR Vol.3,Database,3.96,51
7,R035,Marketing Vol.5,E-Book,3.95,38
8,R076,PhD Thesis Vol.1,Thesis,3.94,17
9,R049,Finance Vol.2,Journal,3.92,24


Average rating by resource type:


,avg_rating
resource_type,
E-Book,3.75
Database,3.83
Journal,3.73
Thesis,3.80
Video,3.69


In [ ]:
# 6. Summary of Key Findings

In [ ]:
summary = pd.DataFrame({
    "question": [
        "Most accessed resource",
        "Most popular resource type",
        "Most active department",
        "Most active user type",
        "Peak month",
        "Busiest weekday",
        "Busiest hour",
        "Underused resources (lowest 10%)",
        "Remote access share (%)",
        "Overall download rate (%)"
    ],
    "result": [
        f"{resource_agg.iloc[0]['resource_id']} ({resource_agg.iloc[0]['access_count']} accesses)",
        type_agg["access_count"].idxmax(),
        department_agg["access_count"].idxmax(),
        user_type_agg["access_count"].idxmax(),
        peak_month,
        peak_day,
        f"{peak_hour}:00",
        len(underused),
        round(remote_share, 1),
        round(df["downloaded"].mean() * 100, 1)
    ]
})

display(summary)

,question,result
0,Most accessed resource,R070 (129 accesses)
1,Most popular resource type,E-Book
2,Most active department,CSE
3,Most active user type,Undergraduate
4,Peak month,April
5,Busiest weekday,Wednesday
6,Busiest hour,16.0:00
7,Underused resources (lowest 10%),10
8,Remote access share (%),40.6
9,Overall download rate (%),32.9


In [ ]:
# 7. Save the Aggregated Tables
### These files can be used in the visualization stage or attached to the project report.

In [ ]:
resource_agg.to_csv("resource_summary.csv", index=False)
department_agg.to_csv("department_summary.csv")
monthly_agg.to_csv("monthly_summary.csv")
weekday_hour_matrix.to_csv("weekday_hour_matrix.csv")
summary.to_csv("key_findings.csv", index=False)

print("Aggregated tables saved.")

Aggregated tables saved.


In [ ]:
# End of Data Aggregation, Representation and Analysis

### The aggregated tables and findings can now be passed to the next stage for Data Visualization, Results and Interpretation.